In [14]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Read the Parquet file
file_path = r"/content/drive/MyDrive/AI/SampleFile/test-product-review.parquet"
print(file_path)
# table = pq.read_table(file_path)
table = pq.read_table(file_path, columns=['review_id','review_body', 'language','stars'],
                      filters=[[('language', '==', 'en')]])
print(table.slice(0,5))

/content/drive/MyDrive/AI/SampleFile/test-product-review.parquet
pyarrow.Table
review_id: string
review_body: string
language: string
stars: int32
----
review_id: [["en_0254382","en_0961337","en_0828526","en_0910562","en_0115007"]]
review_body: [["Well I haven't stunned anyone...yet. So I can only rate what I know. I bought this because it was  (... 235 chars omitted)","I loved the first book so much had to read the sequel. Actually couldn't believe it was even bette (... 16 chars omitted)","This wasn't my favorite bag, but it wasn't my least favorite either. The bag is very long. As a re (... 404 chars omitted)","Good storyline but the characterizations seemed a little lacking and I'm definitely a character re (... 5 chars omitted)","Never noticed a big difference than a normal brush."]]
language: [["en","en","en","en","en"]]
stars: [[2,1,3,3,3]]


In [15]:
# Install if needed:
# pip install pandas pyarrow scikit-learn

import re
import pandas as pd
import pyarrow.parquet as pq

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1) Load the dataset
file_path = r"/content/drive/MyDrive/AI/SampleFile/test-product-review.parquet"

table = pq.read_table(
    file_path,
    columns=["review_id", "review_body", "language", "stars"],
    filters=[[("language", "==", "en")]]
)

df = table.to_pandas()

print(df.head())
print(df.shape)

    review_id                                        review_body language  \
0  en_0254382  Well I haven't stunned anyone...yet. So I can ...       en   
1  en_0961337  I loved the first book so much had to read the...       en   
2  en_0828526  This wasn't my favorite bag, but it wasn't my ...       en   
3  en_0910562  Good storyline but the characterizations seeme...       en   
4  en_0115007  Never noticed a big difference than a normal b...       en   

   stars  
0      2  
1      1  
2      3  
3      3  
4      3  
(460, 4)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
# 2) Data cleaning and label preparation

df = df[["review_id", "review_body", "stars"]].dropna().copy()

# Convert stars to numeric
df["stars"] = pd.to_numeric(df["stars"], errors="coerce")
df = df.dropna(subset=["stars"]).reset_index(drop=True)

# Keep only clear sentiment examples
def create_sentiment_label(stars):
    if stars >= 4:
        return "positive"
    elif stars <= 2:
        return "negative"
    else:
        return "neutral"

df["sentiment"] = df["stars"].apply(create_sentiment_label)

# Remove neutral examples for a simpler binary sentiment task
df = df[df["sentiment"] != "neutral"].reset_index(drop=True)

print(df["sentiment"].value_counts())

sentiment
negative    188
positive    182
Name: count, dtype: int64


In [18]:
# 3) Text preprocessing

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_review"] = df["review_body"].apply(clean_text)

# Optional: drop empty reviews after cleaning
df = df[df["clean_review"] != ""].reset_index(drop=True)

print(df[["review_body", "clean_review", "sentiment"]].head())

                                         review_body  \
0  Well I haven't stunned anyone...yet. So I can ...   
1  I loved the first book so much had to read the...   
2               Loved this!!! True to size and gorg!   
3  I did not receive this item, what steps to tak...   
4  Had this bowl for only 3 months and used it on...   

                                        clean_review sentiment  
0  well i haven t stunned anyone yet so i can onl...  negative  
1  i loved the first book so much had to read the...  negative  
2                   loved this true to size and gorg  positive  
3  i did not receive this item what steps to take...  negative  
4  had this bowl for only months and used it only...  negative  


In [19]:
# 4) Train/Test split

X = df["clean_review"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Class distribution in train:", y_train.value_counts().to_dict())

Train size: 296
Test size: 74
Class distribution in train: {'negative': 150, 'positive': 146}
